# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prjena987/Flyrank-Ai-Starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Selected Lane:** Lane 1 — Content Refresh & Decay Prioritization  
**ML Task Type:** Binary Classification (with downstream decision-support probability scoring)

**Why:** The goal is to predict whether a specific page URL will experience significant organic traffic decay in an upcoming timeframe. Rather than predicting exact raw impression numbers, framing this as a binary classification problem ($1 = \text{decaying}, 0 = \text{stable/growing}$) allows us to output a probability score $P(\text{decay})$. This probability directly powers a decision-support queue, allowing editorial teams to prioritize interventions where ROI is highest.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Predicted Target:** `is_decaying` (Binary Indicator: 1 or 0)  
**Label Origin:** Defined rule over observed performance outcomes (proxy label).

A page is labeled as decaying (`is_decaying = 1`) if its observed organic impression volume over the most recent 30-day period drops by 30% or more compared to its previous 30-day baseline window:

$$\text{Impression Ratio} = \frac{\text{Impressions}_{\text{recent 30d}}}{\text{Impressions}_{\text{baseline 30d}}}$$

If $\text{Impression Ratio} \le 0.70$, then `is_decaying = 1`; otherwise `0`.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

# Target proxy calculation demonstration
df_proxy = pd.DataFrame(
    {
        "url": ["/blog/post-a", "/blog/post-b", "/blog/post-c"],
        "impressions_baseline_30d": [10000, 5000, 2000],
        "impressions_recent_30d": [6500, 4800, 1100],
    }
)

df_proxy["impression_ratio"] = (
    df_proxy["impressions_recent_30d"] / df_proxy["impressions_baseline_30d"]
)
df_proxy["is_decaying"] = (df_proxy["impression_ratio"] <= 0.70).astype(int)

print("Target Proxy Label Derivation:")
print(
    df_proxy[
        [
            "url",
            "impressions_baseline_30d",
            "impressions_recent_30d",
            "impression_ratio",
            "is_decaying",
        ]
    ]
)

Target Proxy Label Derivation:
            url  impressions_baseline_30d  impressions_recent_30d  \
0  /blog/post-a                     10000                    6500   
1  /blog/post-b                      5000                    4800   
2  /blog/post-c                      2000                    1100   

   impression_ratio  is_decaying  
0              0.65            1  
1              0.96            0  
2              0.55            1  


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric:** PR-AUC (Precision-Recall Area Under Curve)  
**Defensible Target:** $\text{PR-AUC} \ge 0.65$ (against a baseline prevalence of ~0.20).

**Why:** Decay events exhibit class imbalance—most healthy pages do not collapse at any given time. ROC-AUC can produce misleadingly high performance figures when true negatives dominate. PR-AUC focuses specifically on precision across recall levels, ensuring we minimize false alarms before sending candidates to human editors.

**Operational Metric:** Precision@50 $\ge 0.75$ (at least 75% of the top 50 flagged pages demonstrate measured performance loss).


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import auc, precision_recall_curve

# Metric calculation stub
y_true = np.array([1, 0, 0, 1, 0, 1, 0, 0, 1, 0])
y_scores = np.array([0.88, 0.12, 0.25, 0.79, 0.31, 0.72, 0.10, 0.42, 0.65, 0.18])

precision, recall, _ = precision_recall_curve(y_true, y_scores)
pr_auc_score = auc(recall, precision)

print(f"Observed Prevalence Baseline: {y_true.mean():.2%}")
print(f"Measured PR-AUC Score: {pr_auc_score:.4f}")

Observed Prevalence Baseline: 40.00%
Measured PR-AUC Score: 1.0000


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** **One row = One unique content URL**.

The dataset aggregates page-level metrics over 30-day windows, combining search performance metrics, page age, and ranking positions into a single analysis table.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load dataset slice (or construct fallback schema matching project structure)
try:
  df = pd.read_csv("../data/starter_data.csv")
except FileNotFoundError:
  np.random.seed(42)
  n_samples = 200
  df = pd.DataFrame({
      "page_id": [f"page_{i}" for i in range(1, n_samples + 1)],
      "url": [f"/blog/article-{i}" for i in range(1, n_samples + 1)],
      "days_since_publish": np.random.randint(15, 730, size=n_samples),
      "impressions_baseline_30d": np.random.randint(200, 15000, size=n_samples),
      "impressions_recent_30d": np.random.randint(50, 15000, size=n_samples),
      "avg_position": np.random.uniform(1.2, 45.0, size=n_samples),
  })

# Derive target
df["impression_ratio"] = df["impressions_recent_30d"] / (
    df["impressions_baseline_30d"] + 1e-5
)
df["is_decaying"] = (df["impression_ratio"] <= 0.70).astype(int)

print(f"Dataset Size: {len(df)} rows")
print("Unit of Analysis: 1 Row = 1 Unique URL")
print(f"Measured Decay Rate (Prevalence): {df['is_decaying'].mean():.2%}\n")

cols_to_show = [
    "url",
    "days_since_publish",
    "impressions_baseline_30d",
    "impressions_recent_30d",
    "avg_position",
    "is_decaying",
]
df[cols_to_show].head()

Dataset Size: 200 rows
Unit of Analysis: 1 Row = 1 Unique URL
Measured Decay Rate (Prevalence): 38.00%



,url,days_since_publish,impressions_baseline_30d,impressions_recent_30d,avg_position,is_decaying
0,/blog/article-1,117,7480,749,39.183313,1
1,/blog/article-2,450,1836,5138,29.074478,0
2,/blog/article-3,285,3896,9381,36.281579,0
3,/blog/article-4,121,11391,240,30.859973,1
4,/blog/article-5,86,11544,10542,26.313476,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML Beats a Fixed Rule:**

1. **Non-Linear Interactions:** Fixed rules like `IF age > 180 AND impression_drop > 30%` fail on younger articles experiencing sudden topic shifts or older evergreen content maintaining stable search demand.
2. **Seasonal Noise:** Fixed thresholds misinterpret expected seasonal drops (such as holiday quiet periods) as true content decay, leading to wasted editing effort.
3. **Continuous Decision Support:** Hard rules provide binary triggers without confidence estimates. ML outputs calibrated probabilities $P(\text{decay})$, allowing content leads to rank recommendations by expected impact when editing capacity is constrained.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Demonstration of static rule blind spots vs actual decay indicators
rigid_rule_flagged = df[
    (df["days_since_publish"] > 180) & (df["impression_ratio"] <= 0.70)
]
young_decaying = df[
    (df["days_since_publish"] <= 180) & (df["is_decaying"] == 1)
]

print(f"Total dataset rows: {len(df)}")
print(f"Actual decaying pages in dataset: {df['is_decaying'].sum()}")
print(f"Pages flagged by rigid age rule (>180d): {len(rigid_rule_flagged)}")
print(
    f"Decaying pages MISSED by rigid age rule (young decay):"
    f" {len(young_decaying)}"
)


Total dataset rows: 200
Actual decaying pages in dataset: 76
Pages flagged by rigid age rule (>180d): 56
Decaying pages MISSED by rigid age rule (young decay): 20


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.